# Project: Data Cleaning Pipeline

Build a reusable data cleaning pipeline handling missing values, outliers, duplicates, inconsistent formats, and validation.

In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
print('Pandas version:', pd.__version__)

## Create Messy Dataset

In [ ]:
np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'id': range(n),
    'name': ['Person_'+str(i) for i in range(n)],
    'age': np.where(np.random.random(n) < 0.08, np.nan,
                    np.where(np.random.random(n) < 0.05, -1,
                             np.random.normal(35, 12, n).astype(int))),
    'salary': np.where(np.random.random(n) < 0.1, np.nan,
                       np.round(np.random.lognormal(10.5, 0.6, n), 2)),
    'department': np.random.choice(['Engineering','Marketing','Sales','HR',None], n),
})
dup_idx = np.random.choice(n, 30, replace=False)
df = pd.concat([df, df.iloc[dup_idx]], ignore_index=True)
df.loc[np.random.choice(n, 10, replace=False), 'salary'] = np.random.uniform(500000, 1000000, 10)
df = df.sample(frac=1).reset_index(drop=True)
print(f'Dataset: {df.shape[0]} rows x {df.shape[1]} cols')
df.head()

## Step 1: Quality Report

In [ ]:
def quality_report(df):
    report = []
    for col in df.columns:
        n_missing = df[col].isnull().sum()
        info = {'column': col, 'dtype': df[col].dtype,
                'missing': n_missing, 'missing_pct': round(n_missing/len(df)*100, 2)}
        if pd.api.types.is_numeric_dtype(df[col]):
            Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
            IQR = Q3 - Q1
            outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
            info['outliers'] = outliers
        report.append(info)
    return pd.DataFrame(report)

print(quality_report(df).to_string(index=False))

## Step 2-4: Handle Issues

In [ ]:
df_clean = df.drop_duplicates().reset_index(drop=True)
print(f'Duplicates removed: {len(df) - len(df_clean)}')

age_bad = (df_clean['age'] < 0) | (df_clean['age'] > 120)
df_clean.loc[age_bad, 'age'] = np.nan
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median()).astype(int)
print(f'Invalid ages fixed: {age_bad.sum()}')

Q1, Q3 = df_clean['salary'].quantile(0.25), df_clean['salary'].quantile(0.75)
upper = Q3 + 3 * (Q3 - Q1)
outliers = df_clean['salary'] > upper
df_clean.loc[outliers, 'salary'] = upper
print(f'Salary outliers capped: {outliers.sum()}')

for col in ['salary', 'department']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0] if col == 'department' else df_clean[col].median())
print(f'Remaining missing: {df_clean.isnull().sum().sum()}')

## Final Validation

In [ ]:
assert df_clean['age'].between(0, 120).all(), 'Age validation failed'
assert (df_clean['salary'] > 0).all(), 'Salary validation failed'
print('='*60)
print('CLEANING SUMMARY'.center(60))
print('='*60)
print(f'Original: {len(df)} rows')
print(f'Final: {len(df_clean)} rows')
print(f'Total fixes: {(len(df)-len(df_clean)) + int(age_bad.sum()) + int(outliers.sum())}')
print(df_clean.describe().round(2))

## Summary

- Quality reporting with issue detection
- Duplicate removal
- Invalid value handling
- Outlier capping
- Missing value imputation
- Validation assertions